# Data Preparation

## Imports

In [1]:
import os
import numpy as np
import pandas as pd

## Database

In [2]:
DATABASE = '../data/raw/Composite_Lap_Joint_Database.xlsx'
OUT = '../data/processed'

master = pd.read_excel(DATABASE, sheet_name='Master_Cleaned')
label_maps = pd.read_excel(DATABASE, sheet_name='Label_Encoder_Maps')
master.shape

(1516, 83)

In [3]:
print(master['Paper_ID'].nunique(), 'studies')
master[['Joint_Material_Class', 'Adherend1_Class', 'Adherend2_Class']].value_counts().head(10)

74 studies


Joint_Material_Class  Adherend1_Class           Adherend2_Class         
Composite-Composite   CFRP                      CFRP                        793
                      GFRP                      GFRP                        216
Composite-Metal       CFRP                      Titanium                    107
                                                Aluminium                    92
                                                Steel                        80
                      GFRP                      Aluminium                    62
                                                Steel                        54
Composite-Composite   BFRP                      BFRP                         28
Composite-Metal       BFRP                      Aluminium                    15
Composite-Composite   Hybrid glass/natural FRP  Hybrid glass/natural FRP     12
Name: count, dtype: int64

## Label Encoding

In [4]:
encoders = {col: dict(zip(g['Category'], g['Code'])) for col, g in label_maps.groupby('Column')}
list(encoders)

['Adherend1_Class',
 'Adherend2_Class',
 'Adhesive_Class',
 'Adhesive_Form',
 'Bonding_Route',
 'Failure_Mode_Class',
 'Fiber_Orientation_Class',
 'Filler1_Type',
 'Filler2_Type',
 'Fillet',
 'Joint_Type_Class',
 'Primer_Used']

## Input Features

In [5]:
FEATURES_CFRP = [
    'No_Plies', 'Adherend1_Thickness_mm', 'Adherend2_Thickness_mm', 'Adherend1_Modulus_GPa', 'Adherend2_Modulus_GPa',
    'Adherend1_Strength_MPa', 'Adherend2_Strength_MPa', 'Filler1_wtpct', 'Filler2_wtpct', 'Mix_Ratio_A_per_B',
    'Adhesive_Modulus_GPa', 'Adhesive_Strength_MPa', 'Adhesive_Elongation_pct', 'Overlap_Length_mm',
    'Joint_Width_mm', 'Adhesive_Thickness_mm', 'Cure_Temp_C', 'Cure_Duration_h', 'PostCure_Temp_C',
    'PostCure_Duration_h', 'Test_Temp_C', 'Loading_Rate_mm_min', 'Humidity_pctRH', 'Aging_Days', 'Adherend1_Class',
    'Adherend2_Class', 'Fiber_Orientation_Class', 'Adhesive_Class', 'Adhesive_Form', 'Filler1_Type', 'Filler2_Type',
    'Joint_Type_Class', 'Fillet', 'Primer_Used', 'Surf_Solvent_detergent_clean',
    'Surf_Abrasion_sanding_grinding_brushing', 'Surf_Grit_sand_blasting', 'Surf_Peel_ply', 'Surf_Plasma_corona',
    'Surf_UV_ozone', 'Surf_Laser', 'Surf_Chemical_electrochemical', 'Surf_Other_modifier', 'Surf_None_as_received',
    'Surf_Ultrasonic_vibration_assisted', 'Surf_Unknown',
]

# composite-composite tasks add the bonding route
FEATURES_COMPOSITE = [
    'No_Plies', 'Adherend1_Thickness_mm', 'Adherend2_Thickness_mm', 'Adherend1_Modulus_GPa', 'Adherend2_Modulus_GPa',
    'Adherend1_Strength_MPa', 'Adherend2_Strength_MPa', 'Filler1_wtpct', 'Filler2_wtpct', 'Mix_Ratio_A_per_B',
    'Adhesive_Modulus_GPa', 'Adhesive_Strength_MPa', 'Adhesive_Elongation_pct', 'Overlap_Length_mm',
    'Joint_Width_mm', 'Adhesive_Thickness_mm', 'Cure_Temp_C', 'Cure_Duration_h', 'PostCure_Temp_C',
    'PostCure_Duration_h', 'Test_Temp_C', 'Loading_Rate_mm_min', 'Humidity_pctRH', 'Aging_Days', 'Adherend1_Class',
    'Adherend2_Class', 'Fiber_Orientation_Class', 'Adhesive_Class', 'Adhesive_Form', 'Filler1_Type', 'Filler2_Type',
    'Joint_Type_Class', 'Fillet', 'Primer_Used', 'Bonding_Route', 'Surf_Solvent_detergent_clean',
    'Surf_Abrasion_sanding_grinding_brushing', 'Surf_Grit_sand_blasting', 'Surf_Peel_ply', 'Surf_Plasma_corona',
    'Surf_UV_ozone', 'Surf_Laser', 'Surf_Chemical_electrochemical', 'Surf_Other_modifier', 'Surf_None_as_received',
    'Surf_Ultrasonic_vibration_assisted', 'Surf_Unknown',
]

META = ['Row_ID', 'Paper_ID', 'Value_Type', 'Target_Is_Derived']
ZERO_IF_MISSING = ['Filler1_wtpct', 'Filler2_wtpct', 'PostCure_Temp_C', 'PostCure_Duration_h', 'Aging_Days']
FAILURE_MODE = {'Adhesive': 0, 'Cohesive': 1, 'Mixed': 2, 'Substrate': 3}

## Joint Sets and Targets

In [6]:
scopes = {
    'CFRP_CFRP': (master['Adherend1_Class'] == 'CFRP') & (master['Adherend2_Class'] == 'CFRP'),
    'Composite_Composite': master['Joint_Material_Class'] == 'Composite-Composite',
}
features = {'CFRP_CFRP': FEATURES_CFRP, 'Composite_Composite': FEATURES_COMPOSITE}

# target column, and the flag that marks targets calculated from load, overlap and width
targets = {
    'MaxLoad': ('MaxLoad_kN', 'Target_Is_Derived_Load'),
    'LSS': ('LSS_MPa', 'Target_Is_Derived_LSS'),
    'FailureMode': ('Failure_Mode_Class', None),
}

tasks = [('CFRP_CFRP', 'MaxLoad'), ('Composite_Composite', 'MaxLoad'),
         ('CFRP_CFRP', 'LSS'), ('Composite_Composite', 'LSS'),
         ('CFRP_CFRP', 'FailureMode'), ('Composite_Composite', 'FailureMode')]

## Build the Six Tables

In [7]:
def encode(series, column):
    if column == 'Adhesive_Form':
        return series.map(lambda v: 0 if pd.isna(v) else encoders[column][v])
    return series.map(lambda v: encoders[column]['Missing'] if pd.isna(v) else encoders[column][v])


summary = []
for scope, target_name in tasks:
    task = f'{scope}__{target_name}'
    target_col, derived_col = targets[target_name]

    sub = master[scopes[scope] & master[target_col].notna()].copy()
    sub['Target_Is_Derived'] = sub[derived_col] if derived_col else 0
    if target_name == 'MaxLoad':
        sub = sub[sub[target_col] > 0]   # a reported load of 0 kN is not used as a target

    table = pd.DataFrame(index=sub.index)
    for col in META + features[scope]:
        s = sub[col]
        if col in ZERO_IF_MISSING:
            s = s.fillna(0)
        table[col] = encode(s, col) if col in encoders else s

    if target_name == 'FailureMode':
        table['Failure_Mode'] = sub[target_col].map(FAILURE_MODE)
    else:
        table[target_col] = sub[target_col]

    table = table.reset_index(drop=True)
    os.makedirs(f'{OUT}/{scope}', exist_ok=True)
    table.to_csv(f'{OUT}/{scope}/{task}.csv', index=False)
    summary.append(dict(Task=task, Records=len(table), Studies=table['Paper_ID'].nunique(),
                        Inputs=table.shape[1] - len(META) - 1, Derived_targets=int(table['Target_Is_Derived'].sum())))
pd.DataFrame(summary)

,Task,Records,Studies,Inputs,Derived_targets
0,CFRP_CFRP__MaxLoad,775,31,46,388
1,Composite_Composite__MaxLoad,1054,45,47,400
2,CFRP_CFRP__LSS,676,28,46,257
3,Composite_Composite__LSS,956,42,47,329
4,CFRP_CFRP__FailureMode,587,29,46,0
5,Composite_Composite__FailureMode,812,42,47,0


## Checks

In [8]:
for scope, target_name in tasks:
    task = f'{scope}__{target_name}'
    t = pd.read_csv(f'{OUT}/{scope}/{task}.csv')
    assert t['Row_ID'].is_unique
    assert t.iloc[:, -1].notna().all()
    print(task, t.shape, 'missing input values:', f"{t.drop(columns=META).iloc[:, :-1].isna().mean().mean():.3f}")

CFRP_CFRP__MaxLoad (775, 51) missing input values: 0.138
Composite_Composite__MaxLoad (1054, 52) missing input values: 0.138
CFRP_CFRP__LSS (676, 51) missing input values: 0.140
Composite_Composite__LSS (956, 52) missing input values: 0.140
CFRP_CFRP__FailureMode (587, 51) missing input values: 0.143
Composite_Composite__FailureMode (812, 52) missing input values: 0.144


## Label Map for Failure Mode

In [9]:
pd.Series(FAILURE_MODE, name='Code')

Adhesive     0
Cohesive     1
Mixed        2
Substrate    3
Name: Code, dtype: int64